# 13 · Do negative and graded ratings improve recommendations?

The current CF + plots + cast + IMDb model treats every MovieLens rating of **4–5 stars (8–10/10)** equally. Lower ratings only prevent the same movie from being recommended again.

This experiment asks whether giving stronger ratings more weight, and subtracting affinities to disliked movies, helps recover users' held-out likes. It runs locally using cached data, with no API calls or additional service cost. It does **not** update the app's selected model.

**Evaluation contract:** canonical per-user chronological 60/20/20 split; features fit using training data; training-catalog candidates; exclude every training-rated movie; evaluate every validation user with at least one rating ≥4; retain all validation positives in Recall@10 and ideal NDCG@10 denominators, including movies outside the candidate catalog. Test feedback is not loaded. This is exploratory validation on a repeatedly used validation set.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "scripts/negative_ratings_experiment.py").is_file(), "Run from this project or its notebooks folder."
sys.path.insert(0, str(ROOT))
(ROOT / "data/.matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "data/.matplotlib"))

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from threadpoolctl import threadpool_limits
from scripts.negative_ratings_experiment import (
    prepare_experiment, run_experiment, rating_weights, profile_coverage,
    PROFILES, PENALTIES, BASELINE,
)
from scripts.export_web_model import file_hash
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 65)


## Prespecified comparisons

| Profile | Positive contribution (MovieLens scale) | Negative contribution |
|---|---|---|
| Binary likes — original control at penalty 0 | 1 for ratings ≥4 | 2.5 − rating for ratings ≤2 |
| Graded likes | rating −3.5 for ratings ≥4 (weights 1:2:3 for 8:9:10/10) | Same explicit dislikes as above |
| All centered ratings | max(rating −3, 0) | max(3 − rating, 0) |

For the centered variant, **6/10 is neutral**, 7–10/10 contributes positively and 1–5/10 negatively. Every rating contributes either to the profile or to seen-item exclusion. This fixed neutral point is an experiment choice, not a claim about all users.

For each profile, test dislike-penalty weights **0, 0.1, 0.25, 0.5, 1**: 15 variants total, specified before execution.

`final score = positive hybrid − penalty × negative hybrid + frozen IMDb adjustment`

Both polarities use the existing CF → plots → cast blend and independent unseen-max normalization. The IMDb prior is added **once**; final signed scores are not clipped or normalized. Relative weights within each polarity matter; multiplying all its ratings by a constant has no effect. With separate normalization, even a single covered dislike can receive full penalty strength, so coverage and the penalty grid matter.

**Controlled scope:** reuse the original binary-like collaborative item basis, plot TF-IDF, cast features, candidates and selected blend weights. This tests profile aggregation; it does not retrain CF on dislikes. With no usable profile, that polarity is zero. The benchmark's original IMDb-only scoring for no-like users is preserved rather than substituting the app's popularity cold start.

Select by macro validation Recall@10, then NDCG@10, then smaller penalty. Compare each selected profile against both the binary control and its own zero-penalty variant so that graded-positive gains are not mistaken for gains from dislikes.


In [ ]:
raw_ratings = np.arange(0.5, 5.01, 0.5)
weight_table = pd.DataFrame({"rating / 10": (2 * raw_ratings).astype(int)})
for profile in PROFILES:
    pos, neg = rating_weights(raw_ratings, profile)
    weight_table[f"{profile}: positive"] = pos
    weight_table[f"{profile}: negative"] = neg
display(weight_table)
print("Prespecified penalty weights:", PENALTIES)


In [ ]:
with threadpool_limits(limits=2):
    context = prepare_experiment(ROOT)

eligible = context["validation"].loc[context["validation"]["rating"] >= 4, "userId"].nunique()
display(pd.DataFrame([{
    "training rating events": len(context["train"]),
    "training users": len(context["users"]),
    "candidate movies": len(context["ids"]),
    "validation users with likes": eligible,
    **{f"frozen {name} weight": value for name, value in context["metadata"]["weights"].items()},
}]))
rating_counts = (context["train"]["rating"].mul(2).astype(int).value_counts()
                 .sort_index().rename_axis("rating / 10").rename("training events").reset_index())
display(rating_counts)

coverage = profile_coverage(context)
coverage_summary = []
for profile in PROFILES:
    for polarity in ["positive", "negative"]:
        column = f"{profile}_{polarity}_count"
        row = {"profile": profile, "polarity": polarity,
               "users with ratings": int(coverage[column].gt(0).sum()),
               "users without ratings": int(coverage[column].eq(0).sum()),
               "median contributing ratings": float(coverage[column].median())}
        for group in context["features"]:
            row[f"users with usable {group}"] = int(
                coverage[f"{profile}_{polarity}_{group}_usable_count"].gt(0).sum())
        coverage_summary.append(row)
display(pd.DataFrame(coverage_summary))
print("Loaded only training feedback and validation labels; verified source hashes.")


## Run the full comparison

The first variant must reproduce notebook 10's saved **per-user** hits, Recall@10 and NDCG@10 to absolute tolerance 1e−12 before any new result is accepted. All variants use the same evaluation users and rank ties by training rating count, then movie ID.

The secondary “observed disliked hits” diagnostic counts top-10 movies with a known validation rating ≤2 (≤4/10). Lower is preferable, but sparse labels make this incomplete: unobserved recommendations are **unknown**, not dislikes. It does not drive model selection.


In [ ]:
result = run_experiment(context=context)
assert result["manifest"]["control_parity"]["verified"]
assert result["manifest"]["test_feedback_read"] is False
assert result["grid"]["users"].nunique() == 1
assert result["grid"]["users"].iloc[0] == eligible
assert len(result["grid"]) == len(PROFILES) * len(PENALTIES)
print("Verified original control for", result["manifest"]["control_parity"]["users"], "users.")
print("Saved experiment artifacts:", result["output"])
display(result["grid"].sort_values(["profile", "penalty_weight"]).reset_index(drop=True))


In [ ]:
grid = result["grid"]
control = grid.loc[grid["variant"] == BASELINE].iloc[0]
selected = result["winners"].copy()
selected["recall change vs control"] = selected["mean_recall_at_10"] - control["mean_recall_at_10"]
selected["relative recall change %"] = 100 * selected["recall change vs control"] / control["mean_recall_at_10"]
display(selected)

best = grid.sort_values(
    ["mean_recall_at_10", "mean_ndcg_at_10", "penalty_weight", "profile"],
    ascending=[False, False, True, True]).iloc[0]
delta = best["mean_recall_at_10"] - control["mean_recall_at_10"]
display(Markdown(
    f"**Highest validation Recall@10:** `{best['variant']}` — "
    f"{best['mean_recall_at_10']:.6f}, versus {control['mean_recall_at_10']:.6f} "
    f"for the original control ({100 * delta / control['mean_recall_at_10']:+.2f}% relative). "
    "**This is a validation-selected result, not an independent test result.**"
))
for row in result["winners"].itertuples():
    zero = grid.loc[grid["variant"] == f"{row.profile}__penalty_0"].iloc[0]
    penalty_gain = row.mean_recall_at_10 - zero["mean_recall_at_10"]
    print(f"{row.profile}: selected penalty {row.penalty_weight:g}; "
          f"Recall gain from dislike penalty alone: {penalty_gain:+.6f}")


In [ ]:
paired = result["paired"]
display(paired.loc[
    (paired["cohort"] == "all validation-positive users") &
    (paired["metric"] == "recall_at_10")
].reset_index(drop=True))
display(paired.loc[
    (paired["cohort"] == "users with training dislikes") &
    (paired["metric"] == "recall_at_10") &
    (paired["comparison"] == "penalty ablation")
].reset_index(drop=True))
display(Markdown(
    "Intervals resample paired users 4,000 times with seed 130. "
    "**They are descriptive and are not adjusted for selecting the best variant on this same data.** "
    "They should not be read as confirmatory significance tests. Gains among users who have "
    "training dislikes help distinguish a useful penalty from effects limited to changing positive weights."
))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4), layout="constrained")
labels = {"binary_likes": "Binary likes", "graded_likes": "Graded likes", "all_centered": "All centered ratings"}
for profile in PROFILES:
    points = grid.loc[grid["profile"] == profile].sort_values("penalty_weight")
    axes[0].plot(points["penalty_weight"], points["mean_recall_at_10"], marker="o", label=labels[profile])
    axes[1].plot(points["penalty_weight"], points["mean_disliked_hits_at_10"], marker="o", label=labels[profile])
axes[0].axhline(control["mean_recall_at_10"], color="gray", linestyle="--", linewidth=1, label="Original control")
axes[0].set(ylabel="Macro Recall@10 (higher is better)", title="Held-out likes recovered")
axes[1].set(ylabel="Mean observed disliked movies in top 10", title="Incomplete dislike diagnostic (lower is better)")
for ax in axes:
    ax.set_xlabel("Dislike penalty weight")
    ax.grid(alpha=0.2)
axes[0].legend(fontsize=8)
chart_path = result["output"] / "penalty_tradeoff.png"
fig.savefig(chart_path, dpi=150)
plt.close(fig)
display(Image(filename=str(chart_path)))
manifest = result["manifest"]
manifest["files"][chart_path.name] = {"sha256": file_hash(chart_path)}
(result["output"] / "experiment_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


## Inspect recommendations

The examples are the first three user IDs with at least three explicit training dislikes, chosen without looking at validation gains. Inspect their training histories beside the binary control and selected profile recommendations. MovieLens ratings are also shown on the app's 1–10 scale.


In [ ]:
example_users = sorted(result["examples"]["userId"].unique())
for user in example_users:
    display(Markdown(f"### User {user}"))
    history = result["histories"].loc[result["histories"]["userId"] == user]
    display(history.sort_values(["rating", "timestamp", "movieId"])
            [["title", "rating_out_of_10"]].head(12).reset_index(drop=True))
    picks = result["examples"].loc[result["examples"]["userId"] == user]
    display(picks.pivot(index="rank", columns="variant", values="title"))


## What this result can establish

- The zero-penalty binary control must reproduce notebook 10, including the benchmark's no-like-user behavior.
- Graded likes at zero penalty test whether distinguishing 8, 9 and 10/10 matters.
- A penalty's benefit is its change relative to **the same profile at zero penalty**, not just relative to the original binary model.
- The centered profile also uses 7/10 as positive evidence and 5/10 as negative evidence. Its neutral point is fixed in advance.
- A small or inconsistent gain warrants further evaluation with fresh users or newly collected chronological feedback. This validation set has been used for selection repeatedly, and notebook 07's historical test results have already been inspected. This notebook makes no new untouched-test claim.
- Current IMDb aggregates can include votes after the MovieLens events, so this remains a retrospective comparison.
- The app retains its selected model. Promotion needs a separate decision after reviewing quality and tradeoffs.

Re-run this notebook top-to-bottom with the existing project kernel, or execute
`.venv/Scripts/python.exe scripts/negative_ratings_experiment.py` from the project root.
All CSV results, per-user comparisons, selected recommendation IDs, profile coverage, deterministic examples, and source hashes are saved under
`data/processed/per_user_chronological_60_20_20_v1/notebook13_negative_ratings/`.
